In [1]:
## Env: allcools
import xarray as xr
import pandas as pd
import numpy as np
# import umap
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
# import scanpy as sc
# import seaborn as sns
import anndata as ad
from sklearn.preprocessing import normalize
from joblib import Parallel, delayed
from pathlib import Path
from collections import Counter
import glob
import math
import cooler
import pathlib
import re
import os
import matplotlib.gridspec as gridspec
from scipy import sparse
import h5py
import anndata
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['font.family'] = 'Arial'
import schicluster
PACKAGE_DIR = schicluster.__path__[0]

In [2]:
cellID_merged = pd.read_csv('/datasets/Public_Datasets/Luo_BICAN_U01_human_brain_dev/snm3C_3C/meta_file/metadata_passQC_05212026.tsv.gz',sep = '\t',index_col=0)
cellID_path = pd.read_csv('/datasets/Public_Datasets/Luo_BICAN_U01_human_brain_dev/snm3C_3C/3C_meta_file_082925.tsv',sep = '\t',index_col=0)
cellID_merged = cellID_merged.merge(cellID_path[['rmbkl_path']], left_index=True, right_index=True,how = 'left') 

/tmp/ipykernel_2611552/3192886318.py:1: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  cellID_merged = pd.read_csv('/datasets/Public_Datasets/Luo_BICAN_U01_human_brain_dev/snm3C_3C/meta_file/metadata_passQC_05212026.tsv.gz',sep = '\t',index_col=0)
/tmp/ipykernel_2611552/3192886318.py:2: DtypeWarning: Columns (3,4,5,79,81,82,84,87,88,89,90,91) have mixed types. Specify dtype option on import or set low_memory=False.
  cellID_path = pd.read_csv('/datasets/Public_Datasets/Luo_BICAN_U01_human_brain_dev/snm3C_3C/3C_meta_file_082925.tsv',sep = '\t',index_col=0)


In [3]:
cellID_merged['age_L3'] = (
    cellID_merged['age_groups'].astype(str) + '_' +
    cellID_merged['L1'].astype(str)+ '-' + cellID_merged['L2'].astype(str)+ '-' + cellID_merged['L3'].astype(str)
)

### Merge 100kb Raw

In [ ]:
leg = {}
chunk_size = 200
outdir = '/tuba/datasets/Public_Datasets/Luo_BICAN_U01_human_brain_dev/snm3C_3C/hic_pseudobulk_raw/age_L3_100kb/'
for cluster, sub_df in cellID_merged.groupby('age_L3'):
    legtmp = []
    # group = cluster.replace(' ', '_').replace('/', '').replace(',', '').replace('.', '')
    tmp = sub_df.copy()
    cluster_dir = f'{outdir}{cluster}'
    if not os.path.exists(cluster_dir):
        os.makedirs(cluster_dir)
    
    tmp = sub_df.copy()
    for i,chunk_start in enumerate(np.arange(0, tmp.shape[0], chunk_size)):
        chunk_dir = f'{outdir}{cluster}_chunk{i}'
        if not os.path.exists(chunk_dir):
            os.makedirs(chunk_dir)
        tmp['rmbkl_path'].iloc[chunk_start:(chunk_start+chunk_size)].to_csv(f'{chunk_dir}/cell_table.tsv', sep='\t', header=False, index=True)
        legtmp.append(f'{cluster}_chunk{i}')
    leg[cluster] = legtmp
    print(cluster, tmp.shape[0])

In [ ]:
f1 = open(f'{outdir}snakemake_cmd_step1.txt', 'w')
f2 = open(f'{outdir}snakemake_cmd_step2.txt', 'w')
for ct in leg:
    for group in leg[ct]:
        cmd = f'hicluster merge-cell-raw --cell_table {outdir}{group}/cell_table.tsv --resolution 100000 --chrom_size_path /tuba/datasets/Public_Datasets/Luo_BICAN_U01_human_brain_dev/snm3C_3C/hg38.chrom_1-22.sizes --output_file {outdir}{group}/raw.cool --chr1 1 --pos1 2 --chr2 3 --pos2 4'
        f1.write(cmd + '\n')
    if len(leg[ct])<2:
        group = leg[ct][0]
        cmd = f'rsync -arv {outdir}{group}/raw.cool {outdir}{ct}/{ct}.raw.cool'
        f2.write(cmd + '\n')
    else:
        cmd = f'cooler merge {outdir}{ct}/{ct}.raw.cool'
        for group in leg[ct]:
            cmd += f' {outdir}{group}/raw.cool'            
        f2.write(cmd + '\n')
        
f1.close()
f2.close()